# 04 - Bangun dan Latih Model

**Tujuan:** Melatih model multi-task (ConvNeXt-Tiny + 4 head) dengan checkpoint tiap epoch dan kemampuan resume.

| | |
|---|---|
| **Input** | `train.csv`, `val.csv`, `labels.json`, arsip gambar |
| **Output** | `checkpoints/best.pt`, `checkpoints/last.pt`, `reports/metrics/history.json` |
| **Prasyarat** | Notebook 03 dan GPU aktif |

> Setiap notebook berdiri sendiri: ia membaca hasil notebook sebelumnya dari Google Drive, jadi sesi Colab boleh diputus/diulang di antara notebook.

In [1]:
# --- Bootstrap (sama di setiap notebook) ---------------------------------------
# 1) Sambungkan Google Drive (tempat semua hasil disimpan permanen)
import os, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

# 2) Lokasi di Drive: folder ini sekaligus ROOT proyek (zip dan semua hasil ada di sini)
ROOT_ON_DRIVE = "/content/drive/MyDrive/models-train/mmfood-multitask"
ZIP_ON_DRIVE  = f"{ROOT_ON_DRIVE}/mmfood-multitask.zip"

#    Folder hasil = ROOT di Drive. Harus diset SEBELUM mengimpor config
os.environ["MMFOOD_PROJECT"] = ROOT_ON_DRIVE

# 3) Ekstrak proyek dari zip ke disk lokal Colab (hanya sekali per sesi)
REPO = "/content/mmfood-multitask"
if not os.path.exists(REPO):
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall("/content")

# 4) Daftarkan folder src/ agar modul proyek bisa di-import
sys.path.insert(0, f"{REPO}/src")

# 5) Muat path folder (P) dan konfigurasi (CFG) dari src/config.py
from config import get_paths, load_cfg
P = get_paths()       # P.processed, P.checkpoints, P.hf_export, ... (folder dibuat otomatis)
CFG = load_cfg(P)     # nilai bawaan + perubahan dari notebook 00
print("Folder hasil:", P.project)

Mounted at /content/drive
Folder hasil: /content/drive/MyDrive/models-train/mmfood-multitask


In [2]:
# Install library yang dibutuhkan (Colab biasanya sudah punya sebagian; -q = senyap)
!pip install -q transformers accelerate huggingface_hub safetensors scikit-learn pillow tqdm matplotlib

## Langkah 4.1 - Muat data dan gambar
Gambar dipulihkan dari arsip Drive ke disk lokal (jika sesi baru), lalu split dan label map dimuat.

In [3]:
import json, math
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from images import restore_images
from data_utils import NUTRITION_KEYS

# Pastikan gambar tersedia di disk lokal (ekstrak dari Drive bila perlu)
n_files = restore_images(P.local_images, f"{P.image_cache}/images.tar")
print("Gambar lokal:", n_files)

# Muat split dan label map hasil notebook 03
train_df = pd.read_csv(f"{P.processed}/train.csv")
val_df = pd.read_csv(f"{P.processed}/val.csv")
LABELS = json.load(open(f"{P.artifacts}/labels.json"))

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"          # mixed precision hanya di GPU
print("Device:", device, "| train:", len(train_df), "| val:", len(val_df))

Gambar lokal: 35998
Device: cuda | train: 28798 | val: 3600


## Langkah 4.2 - DataLoader
`get_transforms` (di `src/dataset.py`): augmentasi untuk train, resize + center-crop untuk val. Mean/std normalisasi diambil dari image processor backbone.

In [4]:
from transformers import AutoImageProcessor
from dataset import get_transforms, make_loader

# Mean/std dan ukuran gambar sesuai backbone pretrained
processor = AutoImageProcessor.from_pretrained(CFG["backbone_name"])
MEAN, STD, SIZE = processor.image_mean, processor.image_std, CFG["img_size"]
json.dump({"image_mean": MEAN, "image_std": STD, "img_size": SIZE}, open(f"{P.artifacts}/preprocess.json", "w"))

train_tf, eval_tf = get_transforms(MEAN, STD, SIZE)
train_dl = make_loader(train_df, P.local_images, train_tf, CFG["batch_size"], True, CFG["num_workers"])
val_dl = make_loader(val_df, P.local_images, eval_tf, CFG["batch_size"], False, CFG["num_workers"])
print("Batch per epoch:", len(train_dl))

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

Batch per epoch: 449


## Langkah 4.3 - Bangun model
Backbone ConvNeXt-Tiny (pretrained ImageNet) + 3 head klasifikasi + 1 head regresi nutrisi. Kelas ada di `src/modeling_mmfood.py` (dipisah dari notebook supaya ikut di-upload ke Hugging Face).

In [5]:
from train_utils import build_model, evaluate

set_seed = lambda s: (torch.manual_seed(s), torch.cuda.manual_seed_all(s), np.random.seed(s))
set_seed(CFG["seed"])   # hasil lebih mudah direproduksi

# Bangun model; bobot backbone diambil dari checkpoint pretrained
model, model_cfg = build_model(CFG, LABELS, device, pretrained_backbone=True)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Parameter: {n_params:.1f} juta")

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  114MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/180 [00:00<?, ?it/s]

[transformers] ConvNextModel LOAD REPORT from: facebook/convnext-tiny-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Parameter: 28.1 juta


## Langkah 4.4 - Optimizer, scheduler, dan resume
- **AdamW** dengan learning rate berbeda: kecil untuk backbone pretrained (jangan merusak fitur yang sudah bagus), besar untuk head baru.
- **Cosine schedule + warmup** 5% awal.
- Jika `last.pt` ada, training **dilanjutkan** dari epoch terakhir.

In [6]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

# Pisahkan parameter backbone dan head agar bisa diberi LR berbeda
head_params = [p for n_, p in model.named_parameters() if not n_.startswith("backbone.")]
opt = AdamW([{"params": model.backbone.parameters(), "lr": CFG["lr_backbone"]},
             {"params": head_params, "lr": CFG["lr_heads"]}], weight_decay=CFG["weight_decay"])

# Jadwal LR: naik linear (warmup) lalu turun mengikuti kurva cosine
total_steps = CFG["epochs"] * len(train_dl)
warm = max(1, int(0.05 * total_steps))
def lr_lambda(step):
    if step < warm:
        return (step + 1) / warm
    return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total_steps - warm)))
sched = LambdaLR(opt, lr_lambda)

# GradScaler untuk mixed precision (nonaktif di CPU)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

# Resume dari checkpoint terakhir bila ada
start_epoch, best_val, history = 0, float("inf"), []
LAST, BEST = f"{P.checkpoints}/last.pt", f"{P.checkpoints}/best.pt"
import os
if os.path.exists(LAST):
    ck = torch.load(LAST, map_location=device)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    start_epoch, best_val, history = ck["epoch"] + 1, ck["best_val"], ck["history"]
    print(f"Resume dari epoch {start_epoch + 1}")
else:
    print("Training dari awal")

Training dari awal


## Langkah 4.5 - Loop training
Setiap epoch: latih -> validasi -> simpan `last.pt` (untuk resume) -> simpan `best.pt` bila val loss membaik. Metrik: akurasi 3 head klasifikasi dan MAE nutrisi (kkal/gram).

In [7]:
for epoch in range(start_epoch, CFG["epochs"]):
    model.train()
    running = 0.0
    pbar = tqdm(train_dl, desc=f"Epoch {epoch + 1}/{CFG['epochs']}")
    for i, b in enumerate(pbar):
        # pindahkan batch ke GPU
        b = {k: v.to(device, non_blocking=True) for k, v in b.items()}
        # forward pass dengan mixed precision
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            out = model(b["pixel_values"], b["food_type"], b["dish"], b["cooking"],
                        b["nutrition"], CFG["loss_weights"])
        # backward + update bobot
        opt.zero_grad(set_to_none=True)
        scaler.scale(out["loss"]).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # cegah gradien meledak
        scaler.step(opt); scaler.update(); sched.step()
        running += out["loss"].item()
        pbar.set_postfix(loss=f"{running / (i + 1):.3f}")

    # validasi di akhir epoch
    val = evaluate(model, val_dl, LABELS, CFG, device)
    val["epoch"] = epoch + 1
    val["train_loss"] = running / len(train_dl)
    history.append(val)
    print({k: round(v, 3) if isinstance(v, float) else v for k, v in val.items()})

    # simpan model terbaik (val loss terendah)
    if val["loss"] < best_val:
        best_val = val["loss"]
        torch.save({"model": model.state_dict(), "epoch": epoch}, BEST)
    # simpan checkpoint lengkap untuk resume
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "epoch": epoch, "best_val": best_val, "history": history}, LAST)
    json.dump(history, open(f"{P.metrics}/history.json", "w"), indent=2)

print("Selesai. Val loss terbaik:", round(best_val, 4))

Epoch 1/6:   0%|          | 0/449 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

{'loss': 2.343, 'acc_food_type': 0.725, 'acc_cooking': 0.607, 'acc_dish': 0.711, 'mae_calories_kcal': 113.692, 'mae_protein_g': 7.336, 'mae_fat_g': 6.097, 'mae_carbohydrate_g': 15.75, 'epoch': 1, 'train_loss': 3.709}


Epoch 2/6:   0%|          | 0/449 [00:00<?, ?it/s]

{'loss': 2.05, 'acc_food_type': 0.745, 'acc_cooking': 0.638, 'acc_dish': 0.744, 'mae_calories_kcal': 104.647, 'mae_protein_g': 7.039, 'mae_fat_g': 5.836, 'mae_carbohydrate_g': 14.119, 'epoch': 2, 'train_loss': 2.028}


... (4 outputs left out)

Epoch 5/6:   0%|          | 0/449 [00:00<?, ?it/s]

{'loss': 1.853, 'acc_food_type': 0.773, 'acc_cooking': 0.648, 'acc_dish': 0.768, 'mae_calories_kcal': 99.183, 'mae_protein_g': 6.654, 'mae_fat_g': 5.522, 'mae_carbohydrate_g': 13.079, 'epoch': 5, 'train_loss': 1.31}


Epoch 6/6:   0%|          | 0/449 [00:00<?, ?it/s]

{'loss': 1.85, 'acc_food_type': 0.772, 'acc_cooking': 0.651, 'acc_dish': 0.768, 'mae_calories_kcal': 98.482, 'mae_protein_g': 6.607, 'mae_fat_g': 5.517, 'mae_carbohydrate_g': 13.027, 'epoch': 6, 'train_loss': 1.245}
Selesai. Val loss terbaik: 1.8498


Lanjut ke notebook **05** untuk evaluasi di test set.